### The following commented code represents benchmarking from the original paper (10 most cited cancer genes)

Generating 10 P-COG results:

In [ ]:
%%bash

mkdir ./refseq_proteomes/cited_genes

echo "NP_004324" > ./refseq_proteomes/cited_genes/BRAF.txt
python3 ./cog.py \
    ./refseq_proteomes/cited_genes/BRAF.txt \
    ./refseq_proteomes/cited_genes/BRAF \
    -t 30

echo "NP_001895" > ./refseq_proteomes/cited_genes/CTNNB1.txt
python3 ./cog.py \
    ./refseq_proteomes/cited_genes/CTNNB1.txt \
    ./refseq_proteomes/cited_genes/CTNNB1 \
    -t 30

echo "NP_005219" > ./refseq_proteomes/cited_genes/EGFR.txt
python3 ./cog.py \
    ./refseq_proteomes/cited_genes/EGFR.txt \
    ./refseq_proteomes/cited_genes/EGFR \
    -t 30

echo "NP_001269316" > ./refseq_proteomes/cited_genes/IDH1.txt
python3 ./cog.py \
    ./refseq_proteomes/cited_genes/IDH1.txt \
    ./refseq_proteomes/cited_genes/IDH1 \
    -t 30

echo "NP_004963" > ./refseq_proteomes/cited_genes/JAK2.txt
python3 ./cog.py \
    ./refseq_proteomes/cited_genes/JAK2.txt \
    ./refseq_proteomes/cited_genes/JAK2 \
    -t 30

echo "NP_000213" > ./refseq_proteomes/cited_genes/KIT.txt
python3 ./cog.py \
    ./refseq_proteomes/cited_genes/KIT.txt \
    ./refseq_proteomes/cited_genes/KIT \
    -t 30

echo "NP_203524" > ./refseq_proteomes/cited_genes/KRAS.txt
python3 ./cog.py \
    ./refseq_proteomes/cited_genes/KRAS.txt \
    ./refseq_proteomes/cited_genes/KRAS \
    -t 30

echo "NP_006209" > ./refseq_proteomes/cited_genes/PIK3CA.txt
python3 ./cog.py \
    ./refseq_proteomes/cited_genes/PIK3CA.txt \
    ./refseq_proteomes/cited_genes/PIK3CA \
    -t 30

echo "NP_000305" > ./refseq_proteomes/cited_genes/PTEN.txt
python3 ./cog.py \
    ./refseq_proteomes/cited_genes/PTEN.txt \
    ./refseq_proteomes/cited_genes/PTEN \
    -t 30

echo "NP_000537" > ./refseq_proteomes/cited_genes/TP53.txt
python3 ./cog.py \
    ./refseq_proteomes/cited_genes/TP53.txt \
    ./refseq_proteomes/cited_genes/TP53 \
    -t 30

ALSO INSTALL MAFFT OR USE ANOTHER ALIGNMENT SOFTWARE!!!

In [ ]:
%%bash

for f in ./refseq_proteomes/cited_genes/*/Results/*.fasta; do mafft --thread 10 "${f}" > "${f}".aln; done

In [ ]:
import json
import re
from glob import glob
import itertools
import math
from collections import Counter
import pandas as pd
import openpyxl
import numpy as np
from Bio import Phylo

In [ ]:
def markov_parse_vis_dataset(text: str):
    # Find the array inside vis.DataSet(...)
    match = re.search(r'vis\.DataSet\s*\(\s*(\[.*?\])\s*\)', text, re.DOTALL)
    if not match:
        # Fallback: look for the first array
        match = re.search(r'\[\s*\{.*?\}\s*\]', text, re.DOTALL)
        if not match:
            return None, None, None
    array_str = match.group(1) if match.lastindex else match.group(0)

    # Extract objects – this simple regex fails on nested braces.
    # Instead, parse with json.loads by first converting JS-like keys to JSON.
    try:
        # Replace single quotes, remove trailing commas, etc.
        clean = re.sub(r'//.*?|/\*.*?\*/', '', array_str)   # remove comments
        # Wrap property names in double quotes (if not already)
        clean = re.sub(r"(\s*?{\s*?|\s*?,\s*?)([a-zA-Z0-9_]+)\s*:", r'\1"\2":', clean)
        clean = re.sub(r"'", '"', clean)   # replace single quotes
        # Remove trailing commas before ] or }
        clean = re.sub(r',\s*(?=[\]}])', '', clean)
        data = json.loads(clean)
    except json.JSONDecodeError:
        return {}, {}, []

    # Use json data
    leaf_grouping = {}
    markov_set = set()
    for obj in data:
        label = obj.get("label")
        if not label:
            continue
        markov = obj.get("markov", 0)
        leaf_grouping[label] = markov
        markov_set.add(markov)
    groups = sorted(markov_set)

    return leaf_grouping, groups

In [ ]:
def mlc_parse_vis_dataset(text: str):
    """
    Group leaves by MLC clusters.

    - Main nodes are those with "color0": "rgb(237,41,57)" (red).
    - Yellow nodes have "color0": "rgb(255,255,0)".
    - For each red node, its group consists of itself and all yellow nodes
      that are directly connected to it by an edge (bidirectional).
    - Leaves with no red/yellow connection remain ungrouped.

    Returns:
        leaf_grouping : dict  {leaf_label: group_id}
        default_colors : dict {group_id: hex_color}
        groups : list of group_id (sorted)
    """
    # Extract the two JavaScript DataSets
    nodes_match = re.search(r'nodes\s*=\s*new\s+vis\.DataSet\s*\(\s*(\[.*?\])\s*\)', text, re.DOTALL)
    edges_match = re.search(r'edges\s*=\s*new\s+vis\.DataSet\s*\(\s*(\[.*?\])\s*\)', text, re.DOTALL)

    if not nodes_match or not edges_match:
        # Fallback: maybe only nodes present (unlikely for MLC)
        return {}, {}, []

    # Helper to clean and parse a JS array of objects
    def parse_js_array(array_str):
        clean = re.sub(r'//.*?|/\*.*?\*/', '', array_str)   # remove comments
        clean = re.sub(r"(\s*?{\s*?|\s*?,\s*?)([a-zA-Z0-9_]+)\s*:", r'\1"\2":', clean)
        clean = re.sub(r"'", '"', clean)
        clean = re.sub(r',\s*(?=[\]}])', '', clean)           # trailing commas
        return json.loads(clean)

    try:
        nodes = parse_js_array(nodes_match.group(1))
        edges = parse_js_array(edges_match.group(1))
    except json.JSONDecodeError:
        return {}, {}, []

    # Build node lookup: id -> {label, color0}
    node_info = {}
    for n in nodes:
        nid = n.get("id")
        if nid is None:
            continue
        label = n.get("label")
        color0 = n.get("color0", "rgb(0,0,0)")
        node_info[nid] = {"label": label, "color0": color0}

    # Build adjacency (undirected)
    adj = {}
    for e in edges:
        f = e.get("from")
        t = e.get("to")
        if f and t:
            adj.setdefault(f, set()).add(t)
            adj.setdefault(t, set()).add(f)

    # Identify red and yellow nodes
    RED = "rgb(237,41,57)"
    YELLOW = "rgb(255,255,0)"
    red_ids = [nid for nid, info in node_info.items() if info["color0"] == RED]
    yellow_ids = set(nid for nid, info in node_info.items() if info["color0"] == YELLOW)

    # Build groups
    leaf_grouping = {}
    group_ids = []

    for red_id in red_ids:
        group_id = red_id
        group_ids.append(group_id)

        # Add red node itself
        red_label = node_info[red_id].get("label")
        if red_label:
            leaf_grouping[red_label] = group_id

        # Collect directly connected yellow neighbors
        for neighbor in adj.get(red_id, []):
            if neighbor in yellow_ids and neighbor not in leaf_grouping:
                neighbor_label = node_info[neighbor].get("label")
                if neighbor_label:
                    leaf_grouping[neighbor_label] = group_id

    groups = sorted(group_ids)

    return leaf_grouping, groups

In [ ]:
def read_fasta(filepath):
    sequences = {}
    current_header = None
    current_seq = []

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                current_header = line[1:].split('_(')[0]
                if not current_header in sequences:
                    sequences[current_header] = ''
            else:
                sequences[current_header] += line.upper()

    return sequences

In [ ]:
def cluster_pairwise_stats(fasta_dict, group_dict):
    groups = {}
    for seq_id, group in group_dict.items():
        if seq_id not in fasta_dict:
            continue
        groups.setdefault(group, []).append(fasta_dict[seq_id])

    results = {}
    for group, seqs in groups.items():
        if len(seqs) < 2:
            results[group] = (None, None)
            continue

        length = len(seqs[0])
        if any(len(s) != length for s in seqs):
            raise ValueError(
                f"Sequences in group {group} have different lengths. "
                "Please align them first (e.g., with MAFFT)."
            )

        identities = []
        for s1, s2 in itertools.combinations(seqs, 2):
            matches = sum(a == b for a, b in zip(s1, s2))
            pid = matches / length
            identities.append(pid)

        mean_pid = sum(identities) / len(identities)
        variance = sum((x - mean_pid) ** 2 for x in identities) / len(identities)
        std_pid = math.sqrt(variance)

        results[group] = (mean_pid, std_pid)

    return results

In [ ]:
def cluster_pairwise_stats_additional(tree_path, group_dict):
    rev_group_dict = dict()

    for k in group_dict:
        if not (group_dict[k] in rev_group_dict):
            rev_group_dict[group_dict[k]] = list()
        rev_group_dict[group_dict[k]].append(k)

    tree = Phylo.read(tree_path, "newick")

    leaves = tree.get_terminals()
    leaf_names = [leaf.name.split('__')[0] for leaf in leaves]

    matrix_data = [
        [tree.distance(leaf1, leaf2) for leaf2 in leaves]
        for leaf1 in leaves
    ]

    mldistdf = pd.DataFrame(matrix_data, index=leaf_names, columns=leaf_names)

    mean_pwds = list()
    mean_iwds = list()
    for k in rev_group_dict:
        for k1 in rev_group_dict:
            valid_elementsk = [x for x in rev_group_dict[k] if x in mldistdf.index]
            valid_elementsk1 = [x for x in rev_group_dict[k1] if x in mldistdf.index]
            sub_df = mldistdf.loc[valid_elementsk, valid_elementsk1]
            if k == k1:
                mean_pwds.append(sub_df.values[np.triu_indices(len(sub_df), k=1)].mean())
            else:
                mean_iwds.append(sub_df.values.mean())

    mean_pwd = sum(mean_pwds) / len(mean_pwds)
    var_pwd = sum((x - mean_pwd) ** 2 for x in mean_pwds) / len(mean_pwds)
    std_pwd = math.sqrt(var_pwd)

    mean_iwd = sum(mean_iwds) / len(mean_iwds)
    var_iwd = sum((x - mean_iwd) ** 2 for x in mean_iwds) / len(mean_iwds)
    std_iwd = math.sqrt(var_iwd)

    return mean_pwd, std_pwd, mean_iwd, std_iwd

In [ ]:
def cluster_pairwise_stats_additional(mldist_path, group_dict):
    rev_group_dict = dict()

    for k in group_dict:
        if not (group_dict[k] in rev_group_dict):
            rev_group_dict[group_dict[k]] = list()
        rev_group_dict[group_dict[k]].append(k.replace(':', '_'))

    with open(mldist_path, "r") as f:
        lines = f.readlines()[1:]

    data = []
    for line in lines:
        parts = line.strip().split()
        sample_name = parts[0].split('__')[0]
        distances = [float(x) for x in parts[1:]]
        data.append([sample_name] + distances)

    headers = [row[0] for row in data]

    mldistdf = pd.DataFrame([row[1:] for row in data], index=headers, columns=headers)

    mean_pwds = []
    mean_iwds = []

    keys = list(rev_group_dict.keys())

    for i in range(len(keys)):
        k = keys[i]

        valid_elementsk = [x for x in rev_group_dict[k] if x in mldistdf.index]

        if len(valid_elementsk) > 1:
            sub_df_intra = mldistdf.loc[valid_elementsk, valid_elementsk]
            intra_values = sub_df_intra.values[np.triu_indices(len(sub_df_intra), k=1)]
            mean_pwds.append(intra_values.mean())
        else:
            mean_pwds.append(0.0)

        for j in range(i + 1, len(keys)):
            k1 = keys[j]

            valid_elementsk1 = [x for x in rev_group_dict[k1] if x in mldistdf.index]

            if len(valid_elementsk) > 0 and len(valid_elementsk1) > 0:
                sub_df_inter = mldistdf.loc[valid_elementsk, valid_elementsk1]

                mean_iwds.append(sub_df_inter.values.mean())

    mean_pwd = sum(mean_pwds) / len(mean_pwds)
    var_pwd = sum((x - mean_pwd) ** 2 for x in mean_pwds) / len(mean_pwds)
    std_pwd = math.sqrt(var_pwd)

    mean_iwd = sum(mean_iwds) / len(mean_iwds)
    var_iwd = sum((x - mean_iwd) ** 2 for x in mean_iwds) / len(mean_iwds)
    std_iwd = math.sqrt(var_iwd)

    return mean_pwd, std_pwd, mean_iwd, std_iwd

In [ ]:
benchmark_data = [['Tool', 'Gene', 'Cluster', 'Size', 'μPID', 'σPID']]

for pyvis_name in glob('./refseq_proteomes/cited_genes/*/Results/*pyvis.html'):
    print(pyvis_name.split('/')[-3])
    with open(pyvis_name, 'r') as pyvis_file:
        pyvis_txt = pyvis_file.read()
    markovs = markov_parse_vis_dataset(pyvis_txt)
    mlcs = mlc_parse_vis_dataset(pyvis_txt)
    fasta_name = pyvis_name.replace('_pyvis.html', '.fasta.aln')
    fasta_dict = read_fasta(fasta_name)
    markov_stats = cluster_pairwise_stats(fasta_dict, markovs[0])
    mlc_stats = cluster_pairwise_stats(fasta_dict, mlcs[0])
    mldist_path = pyvis_name.replace('_pyvis.html', '.fasta.aln.mldist')
    markov_mean_pwd, markov_std_pwd, markov_mean_iwd, markov_std_iwd = cluster_pairwise_stats_additional(mldist_path, markovs[0])
    mlc_mean_pwd, mlc_std_pwd, mlc_mean_iwd, mlc_std_iwd = cluster_pairwise_stats_additional(mldist_path, mlcs[0])
    print("Markov cluster stats:")
    informative_nodes = 0
    pre_weighted_avg = 0
    pre_weighted_std = 0
    for k in markov_stats:
        markov_counter = Counter(list(markovs[0].values()))
        if markov_stats[k][0]:
            # print(f'Cluster {k} size: {markov_counter[k]}, μPID: {markov_stats[k][0]:0.2f}, σPID: {markov_stats[k][1]:0.2f}', end='; ')
            informative_nodes += markov_counter[k]
            pre_weighted_avg += markov_counter[k]*markov_stats[k][0]
            pre_weighted_std += markov_counter[k]*markov_stats[k][1]
        else:
            pass
            # print(f'Cluster {k} size: {markov_counter[k]}', end='; ')
        benchmark_data.append(['P-COGs(Markov)', pyvis_name.split('/')[-3], k, markov_counter[k], markov_stats[k][0], markov_stats[k][1]])
    if informative_nodes == 0:
        print('NA')
    else:
        print(f"1) Total informative nodes using Markov clustering: {informative_nodes}/{len(markovs[0])}")
        print(f"2) Total μPID: {pre_weighted_avg/informative_nodes:0.2f}")
        print(f"3) Total σPID: {pre_weighted_std/informative_nodes:0.2f}")
        print(f"4) Total μPWD: {markov_mean_pwd:0.2f}")
        print(f"5) Total σPWD: {markov_std_pwd:0.2f}")
        print(f"6) Total μICD: {markov_mean_iwd:0.2f}")
        print(f"7) Total σICD: {markov_std_iwd:0.2f}")
    print("MLC cluster stats:")
    informative_nodes = 0
    pre_weighted_avg = 0
    pre_weighted_std = 0
    for k in mlc_stats:
        mlc_counter = Counter(list(mlcs[0].values()))
        if mlc_stats[k][0]:
            # print(f'Cluster {k} size: {mlc_counter[k]}, μPID: {mlc_stats[k][0]:0.2f}, σPID: {mlc_stats[k][1]:0.2f}', end='; ')
            informative_nodes += mlc_counter[k]
            pre_weighted_avg += mlc_counter[k]*mlc_stats[k][0]
            pre_weighted_std += mlc_counter[k]*mlc_stats[k][1]
        else:
            pass
            # print(f'Cluster {k} size: {mlc_counter[k]}', end='; ')
        benchmark_data.append(['P-COGs(MLC)', pyvis_name.split('/')[-3], k, mlc_counter[k], mlc_stats[k][0], mlc_stats[k][1]])
    if informative_nodes == 0:
        print('NA')
    else:
        print(f"1) Total informative nodes using MLCs: {informative_nodes}/{len(markovs[0])}")
        print(f"2) Total μPID: {pre_weighted_avg/informative_nodes:0.2f}")
        print(f"3) Total σPID: {pre_weighted_std/informative_nodes:0.2f}")
        print(f"4) Total μPWD: {mlc_mean_pwd:0.2f}")
        print(f"5) Total σPWD: {mlc_std_pwd:0.2f}")
        print(f"6) Total μICD: {mlc_mean_iwd:0.2f}")
        print(f"7) Total σICD: {mlc_std_iwd:0.2f}")

benchmark_df = pd.DataFrame(benchmark_data[1:], columns=benchmark_data[0])
benchmark_df.to_excel('benchmark.xlsx')